In [3]:
#  Calculate the average number of sentences in the gold summary in the summarisation dataset

from datasets import load_dataset
import nltk
import json

output_key = {
    "ccsum": "summary",
    "summscreen": "output",
    "gov_report": "output",
    "qmsum": "output"
}

input_key = {
    "ccsum": "article",
    "summscreen": "input",
    "gov_report": "input",
    "qmsum": "input"
}

def count_sentences(text: str) -> int:
    """Count the number of sentences in a text using NLTK sentence tokenizer."""
    if not text or not text.strip():
        return 0
    
    sentences = nltk.sent_tokenize(text.strip())
    # Filter out very short "sentences" (likely tokenization errors)
    sentences = [s for s in sentences if len(s.strip()) > 3]
    return len(sentences)


def average_summary_length(data, dataset_name: str) -> float: 
    """Calculate the average number of sentences in the gold summaries."""
    gold_summaries = [item[output_key[dataset_name]] for item in data]
    num_sents = [count_sentences(summary) for summary in gold_summaries]
    avg_num_sents = sum(num_sents) / len(num_sents)
    
    return avg_num_sents, num_sents

def avearge_input_length(data, dataset_name: str) -> float:
    """Calculate the average number of sentences in the input document."""
    docs = [item[input_key[dataset_name]] for item in data]
    num_sents = [count_sentences(doc) for doc in docs]
    avg_num_sents = sum(num_sents) / len(num_sents)

    return avg_num_sents, num_sents


# Load the dataset (this part might change for different datasets)
# data = load_dataset("tau/scrolls", "qmsum", trust_remote_code=True)["validation"]
# max_samples = len(data) + 1
# data = data.select(range(min(max_samples, len(data))))

# Load non query-based QMSum data
data_path = "/mnt/ceph_rbd/datasets/QMSum/processed_data/test.jsonl"
data = []
with open(data_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))
print("Number of instances in QMSum test data", len(data))


# For CCSum dataset
# max_samples = 100000
# ccsum_dataset = load_dataset("/home/xiaotang/Project/context-faithful-llm/datasets/ccsum")
# dataset_abstractive = ccsum_dataset.filter(lambda x: x["abstractiveness_bin"] == "high")
# data = dataset_abstractive['test']
# data = data.select(range(min(max_samples, len(data))))

# avg_num_sents = average_summary_length(data, "summscreen")
# avg_num_sents, num_sents = average_summary_length(data, "ccsum")
avg_num_sents, num_sents = average_summary_length(data, "qmsum")

print(avg_num_sents)
print(num_sents)

# Count how many summaries have more than avg num sents
num_sents_gt_1 = sum(1 for n in num_sents if n > avg_num_sents)
print(f"Number of summaries with the number of sentences more than average: {num_sents_gt_1}")
print(f"Total number of samples: {len(num_sents)}")

avg_num_sents, num_sents = avearge_input_length(data, "qmsum")

print(avg_num_sents)
print(num_sents)

print("Average number of sentences in the input document:", avg_num_sents)

Number of instances in QMSum test data 37
6.135135135135135
[8, 7, 6, 7, 4, 6, 4, 4, 6, 6, 3, 5, 6, 5, 7, 5, 5, 5, 9, 5, 8, 6, 7, 6, 5, 10, 6, 9, 7, 9, 7, 3, 10, 6, 4, 5, 6]
Number of summaries with the number of sentences more than average: 13
Total number of samples: 37
740.4594594594595
[522, 643, 406, 1016, 881, 859, 416, 1338, 265, 1282, 300, 1001, 713, 713, 473, 688, 375, 1636, 796, 1017, 753, 757, 688, 615, 791, 513, 954, 1116, 875, 285, 509, 1004, 1337, 469, 513, 513, 365]
Average number of sentences in the input document: 740.4594594594595


In [4]:
# The maximum token length among the first 300 validation samples in GovReport

from transformers import AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm

def main():
    # Load Qwen3-8B tokenizer
    print("Loading Qwen3-8B tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")  # Use Qwen2.5 as Qwen3 might not be available
    
    # Load GovReport dataset
    print("Loading QMSum dataset...")
    # dataset = load_dataset("tau/scrolls", "qmsum")["validation"]
    # Load non query-based QMSum data
    data_path = "/mnt/ceph_rbd/datasets/QMSum/processed_data/test.jsonl"
    dataset = []
    with open(data_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                dataset.append(json.loads(line))
    print("Number of instances in QMSum test data", len(dataset))
    
    # Take first 300 samples
    # samples = dataset.select(range(min(300, len(dataset))))
    samples = dataset
    print(f"Processing {len(samples)} samples...")
    
    max_length = 0
    max_sample_idx = 0
    lengths = []
    
    # Compute token lengths
    for i, sample in tqdm(enumerate(samples), total=len(samples)):
        # Tokenize the input document
        tokens = tokenizer.encode(sample['input'], add_special_tokens=False)
        token_length = len(tokens)
        lengths.append(token_length)
        
        # Track maximum
        if token_length > max_length:
            max_length = token_length
            max_sample_idx = i
    
    # Print results
    print(f"\n=== Results ===")
    print(f"Max token length: {max_length:,}")
    print(f"Sample with max length: #{max_sample_idx}")
    print(f"Average token length: {sum(lengths)/len(lengths):.1f}")
    print(f"Min token length: {min(lengths):,}")
    
    # Print some statistics
    lengths.sort()
    print(f"\n=== Token Length Distribution ===")
    print(f"50th percentile: {lengths[len(lengths)//2]:,}")
    print(f"75th percentile: {lengths[len(lengths)*3//4]:,}")
    print(f"90th percentile: {lengths[len(lengths)*9//10]:,}")
    print(f"95th percentile: {lengths[len(lengths)*95//100]:,}")
    
    # Show how many samples exceed common limits
    limits = [4096, 8192, 16384, 32768]
    print(f"\n=== Samples exceeding common limits ===")
    for limit in limits:
        count = sum(1 for l in lengths if l > limit)
        percentage = count / len(lengths) * 100
        print(f">{limit:,} tokens: {count}/{len(lengths)} ({percentage:.1f}%)")

main()

Loading Qwen3-8B tokenizer...
Loading QMSum dataset...
Number of instances in QMSum test data 37
Processing 37 samples...


100%|██████████| 37/37 [00:01<00:00, 29.96it/s]



=== Results ===
Max token length: 33,230
Sample with max length: #17
Average token length: 13608.6
Min token length: 3,797

=== Token Length Distribution ===
50th percentile: 11,641
75th percentile: 17,867
90th percentile: 21,817
95th percentile: 26,917

=== Samples exceeding common limits ===
>4,096 tokens: 36/37 (97.3%)
>8,192 tokens: 30/37 (81.1%)
>16,384 tokens: 12/37 (32.4%)
>32,768 tokens: 1/37 (2.7%)


In [5]:
# Load non query-based QMSum test data
import json

data_path = "/mnt/ceph_rbd/datasets/QMSum/processed_data/test.jsonl"
data = []
with open(data_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))
print("Number of instances in QMSum test data", len(data))
print(len(data[:10]))
print(data[1].keys())

Number of instances in QMSum test data 37
10
dict_keys(['input', 'output'])


In [12]:
# 构造prompt

doc = "{that's} blabla {dismaker}"
idx = 0
prompt_template = f"Summary the following document:{doc}" + f"{idx}.{doc}"

print(prompt_template)

Summary the following document:{that's} blabla {dismaker}0.{that's} blabla {dismaker}


In [1]:
# Given the path to a prediction file, compute the average length of the generated summary (number of sentences)
import json
import nltk

def count_sentences(text: str) -> int:
    """Count the number of sentences in a text using NLTK sentence tokenizer."""
    if not text or not text.strip():
        return 0
    
    sentences = nltk.sent_tokenize(text.strip())
    # Filter out very short "sentences" (likely tokenization errors)
    sentences = [s for s in sentences if len(s.strip()) > 3]
    return len(sentences)


def average_summary_length(data) -> float: 
    """Calculate the average number of sentences in the gold summaries."""
    pred_summaries = [item["generated_summary"] for item in data]
    num_sents = [count_sentences(summary) for summary in pred_summaries]
    avg_num_sents = sum(num_sents) / len(num_sents)
    
    return avg_num_sents, num_sents

# data_path = "/mnt/ceph_rbd/project/context-faithful-llm/long-form/results/attribution/qwen3-32b_qmsum_validation_272_gen_attr_sentence_num30.json"
data_path = "/mnt/ceph_rbd/project/context-faithful-llm/long-form/results/summary/qwen3-8b_gov_report_validation_300_sum_cot.json"
with open(data_path, 'r') as fin:
    data = json.load(fin)


print(len(data))
avg_num_sents, num_sents = average_summary_length(data)
print(avg_num_sents)
print(num_sents)


300
7.83
[6, 7, 7, 7, 6, 9, 5, 5, 7, 6, 14, 10, 5, 6, 7, 8, 7, 6, 11, 6, 8, 7, 5, 8, 6, 8, 7, 6, 8, 8, 7, 7, 8, 9, 5, 6, 9, 7, 14, 10, 6, 6, 6, 14, 6, 8, 5, 8, 8, 10, 7, 8, 10, 7, 10, 7, 9, 6, 11, 5, 9, 7, 8, 8, 13, 9, 7, 7, 7, 5, 6, 5, 5, 9, 7, 6, 8, 10, 8, 7, 11, 8, 10, 6, 9, 7, 11, 11, 9, 11, 6, 10, 7, 8, 6, 11, 8, 6, 6, 7, 9, 8, 9, 11, 8, 6, 12, 9, 7, 7, 7, 4, 9, 9, 6, 6, 7, 9, 18, 7, 7, 6, 9, 9, 6, 8, 4, 9, 7, 9, 8, 7, 8, 8, 12, 9, 7, 7, 6, 9, 11, 8, 7, 12, 7, 7, 10, 8, 8, 6, 9, 9, 10, 9, 10, 6, 7, 9, 5, 8, 9, 6, 9, 8, 6, 8, 7, 8, 6, 6, 7, 7, 11, 12, 6, 7, 8, 9, 9, 9, 8, 11, 5, 8, 8, 7, 8, 7, 8, 8, 7, 8, 7, 9, 16, 10, 6, 9, 6, 7, 10, 12, 7, 8, 6, 10, 7, 9, 12, 9, 11, 7, 9, 6, 8, 9, 5, 10, 6, 6, 8, 10, 8, 9, 7, 11, 6, 6, 6, 7, 7, 4, 5, 7, 9, 8, 6, 7, 8, 7, 8, 6, 10, 8, 10, 6, 10, 6, 6, 7, 6, 9, 9, 9, 8, 6, 8, 8, 9, 6, 8, 13, 7, 6, 12, 7, 5, 9, 9, 6, 9, 6, 7, 6, 6, 7, 6, 5, 6, 6, 8, 5, 7, 10, 6, 8, 9, 9, 8, 5, 5, 8, 8, 6, 10, 9, 7, 8, 7, 8]


In [1]:
# Given the path to a prediction file, compute the average length of the generated summary (number of tokens)

import json
from transformers import AutoTokenizer

def compute_average_summary_length(data, model_name="Qwen/Qwen3-8B"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    token_counts = []
    for sample in data:
        tokens = tokenizer.encode(sample['generated_summary'], add_special_tokens=False)
        token_counts.append(len(tokens))
    
    avg_token_count = sum(token_counts) / len(token_counts) if token_counts else 0
    return avg_token_count, token_counts


# data_path = "/mnt/ceph_rbd/project/context-faithful-llm/long-form/results/attribution/qwen3-32b_qmsum_validation_272_gen_attr_sentence_num30.json"
data_path = "/mnt/ceph_rbd/project/context-faithful-llm/long-form/results/summary/qwen3-8b_gov_report_validation_300_attr-sent30_base+impt_prefix.json"
with open(data_path, 'r') as fin:
    data = json.load(fin)

print(len(data))
avg_token_count, token_counts = compute_average_summary_length(data)
print(avg_token_count)
print(token_counts)

/mnt/ceph_rbd/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


300
613.1366666666667
[666, 898, 579, 352, 596, 453, 702, 795, 650, 445, 841, 865, 620, 841, 655, 687, 418, 751, 828, 459, 772, 545, 657, 801, 709, 638, 474, 615, 341, 561, 293, 374, 781, 930, 343, 736, 969, 860, 631, 366, 534, 446, 672, 332, 501, 439, 483, 364, 381, 605, 485, 365, 552, 393, 552, 730, 514, 435, 516, 360, 676, 997, 542, 578, 576, 446, 641, 402, 488, 523, 443, 567, 383, 609, 346, 386, 509, 665, 792, 662, 541, 702, 451, 402, 918, 801, 611, 1022, 749, 795, 402, 779, 314, 675, 465, 655, 1021, 627, 641, 550, 777, 405, 547, 589, 1022, 911, 915, 420, 1022, 358, 397, 475, 606, 638, 258, 710, 368, 758, 891, 672, 676, 636, 753, 736, 610, 957, 640, 720, 601, 482, 579, 618, 481, 816, 803, 868, 684, 410, 526, 1022, 603, 553, 573, 918, 659, 711, 467, 626, 673, 638, 373, 660, 496, 763, 690, 704, 489, 618, 476, 652, 309, 466, 412, 736, 798, 388, 432, 731, 809, 613, 1014, 1022, 687, 367, 835, 513, 502, 690, 563, 587, 548, 817, 367, 569, 586, 312, 831, 496, 920, 601, 727, 304, 493, 1024,

In [3]:
datasets = ["gov_report", "qmsum"]
dataset_lens = {
    "gov_report": 300,
    "qmsum": 272
}
# models = ["llama3.1-8b", "qwen3-8b", "qwen3-32b"]
# exps = ["base", "sum_cot", "attr-sent30_base+impt", "attr-sent30_base+impt_prefix"]
models = ["llama3.1-8b", "qwen3-8b"]
exps = ["attr-cc_sent30_base+impt", "attr-cc_sent30_base+impt_prefix"]


# for dataset in datasets:
#     for model in models:
#         for exp in exps:
#             data_path = f"/mnt/ceph_rbd/project/context-faithful-llm/long-form/results/summary/{model}_{dataset}_validation_{dataset_lens[dataset]}_{exp}.json"
#             print(f"Dataset: {dataset}, Model: {model}, Exp: {exp}")
#             with open(data_path, 'r') as fin:
#                 data = json.load(fin)
#             avg_token_count, token_counts = compute_average_summary_length(data)
#             print(avg_token_count)
#             print(token_counts)

# for dataset in datasets:
#     for model in models:
#         # Generative attribution
#         data_path = f"/mnt/ceph_rbd/project/context-faithful-llm/long-form/results/attribution/{model}_{dataset}_validation_{dataset_lens[dataset]}_gen_attr_sentence_num30.json"
#         print(f"Dataset: {dataset}, Model: {model}, Exp: gen_attr")
#         with open(data_path, 'r') as fin:
#             data = json.load(fin)
#         avg_token_count, token_counts = compute_average_summary_length(data)
#         print(avg_token_count)
#         print(token_counts)


num_sents = [5, 10, 20, 30, 40, 50]
for num_sent in num_sents:
    data_path = f"/mnt/ceph_rbd/project/context-faithful-llm/long-form/results/ablation/qwen3-8b_gov_report_train_100_attr-sent{num_sent}_base+impt.json"
    with open(data_path, 'r') as fin:
        data = json.load(fin)
    print("Num of sentences:", num_sent)
    avg_token_count, token_counts = compute_average_summary_length(data)
    print(avg_token_count)
    print(token_counts)

Num of sentences: 5
337.27
[316, 419, 269, 373, 223, 398, 206, 229, 282, 432, 190, 448, 397, 466, 389, 375, 233, 209, 661, 276, 250, 301, 468, 320, 483, 487, 530, 226, 253, 418, 331, 494, 273, 252, 544, 258, 648, 390, 203, 196, 223, 176, 475, 447, 461, 264, 450, 537, 258, 311, 393, 346, 287, 216, 387, 224, 235, 338, 618, 383, 447, 575, 477, 233, 238, 276, 305, 235, 366, 605, 445, 210, 258, 313, 257, 351, 387, 286, 389, 236, 176, 224, 257, 200, 215, 300, 260, 256, 380, 369, 366, 346, 170, 279, 242, 189, 242, 392, 463, 367]
Num of sentences: 10
437.42
[510, 441, 346, 582, 555, 402, 260, 323, 544, 814, 282, 496, 529, 509, 474, 651, 498, 348, 869, 400, 383, 291, 577, 514, 474, 398, 449, 534, 470, 546, 585, 493, 364, 559, 392, 365, 393, 477, 360, 300, 338, 246, 539, 322, 433, 434, 635, 505, 426, 581, 359, 245, 387, 338, 316, 338, 415, 447, 413, 452, 496, 713, 571, 341, 383, 325, 445, 386, 593, 419, 528, 394, 309, 477, 400, 332, 315, 362, 508, 254, 542, 328, 426, 359, 274, 294, 688, 266, 429